# 7. Modelos:  BPE, Embeddings, Neural LM

<a target="_blank" href="https://colab.research.google.com/github/umoqnier/cl-2026-2-lab/blob/main/notebooks/7_neural_lm.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Objetivos

- Entrenar modelos para sub-word tokenization
  - Aplicar BPE a corpus
- Entrenar modelos para *embeddings*
  - Word2Vec
    - Skip gram
    - CBow
- Implementación de modelo del lenguaje Neuronal de Bengio
  - Generación de lenguaje

In [7]:
import os
import re
from rich import print as rprint
from nltk import word_tokenize
from collections import Counter
from nltk.stem.snowball import SnowballStemmer

## Funciones de preprocesamiento

In [8]:
import unicodedata


def strip_accents(s: str) -> str:
    """Remove diacritical marks from characters in a Unicode string.

    Uses Unicode NFD (Normalization Form Decomposition) normalization to decompose accented characters into their
    base character + combining mark, then filters out combining marks (Mark, Nonspacing, Mn category).
    """
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )


def preprocess_text(text: str, to_lower: bool = True) -> str:
    """Preprocess text by normalizing, lowercasing and removing extra spaces.
    """
    # 1. Unicode Normalization (NFC)
    text = unicodedata.normalize("NFC", text)

    if to_lower:
        text = text.lower()

    # 3. Collapse all whitespace/newlines into a single space
    text = re.sub(r"\s+", " ", text)

    # 4. Clean up leading/trailing whitespace
    text = text.strip()

    return text

## Byte-Pair Encoding

### Vamos a tokenizar 🌈
![](https://i.pinimg.com/736x/58/6b/88/586b8825f010ce0e3f9c831f568aafa8.jpg)

In [2]:
BASE_PATH = "."
CORPORA_PATH = f"{BASE_PATH}/data/"
MODELS_PATH = f"{BASE_PATH}/models/"

os.makedirs(CORPORA_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

In [24]:
TOKENIZERS_DATA_PATH = f"{CORPORA_PATH}/tokenization"
TOKENIZERS_MODEL_PATH = f"{MODELS_PATH}/sub-word"

os.makedirs(TOKENIZERS_DATA_PATH, exist_ok=True)
os.makedirs(TOKENIZERS_MODEL_PATH, exist_ok=True)

### Corpus en español: Wikipedia

In [9]:
from datasets import load_dataset, load_dataset_builder

In [10]:
data_builder = load_dataset_builder("wikimedia/wikipedia", "20231101.es")

In [11]:
rprint(data_builder.info)

DatasetInfo(
    description='',
    citation='',
    homepage='',
    license='',
    features={
        'id': Value(dtype='string', id=None),
        'url': Value(dtype='string', id=None),
        'title': Value(dtype='string', id=None),
        'text': Value(dtype='string', id=None)
    },
    post_processed=None,
    supervised_keys=None,
    builder_name='parquet',
    dataset_name='wikipedia',
    config_name='20231101.es',
    version=0.0.0,
    splits={
        'train': SplitInfo(
            name='train',
            num_bytes=6033536133,
            num_examples=1841155,
            shard_lengths=None,
            dataset_name=None
        )
    },
    download_checksums=None,
    download_size=3493595869,
    post_processing_size=None,
    dataset_size=6033536133,
    size_in_bytes=None
)

In [12]:
dataset = load_dataset(
    "wikimedia/wikipedia", "20231101.es", split="train", streaming=True
)

In [13]:
wiki_words = []
for article in dataset.take(1):
    rprint(preprocess_text(article["text"][:1000], to_lower=False))

Andorra, oficialmente Principado de Andorra () es un micro-Estado soberano sin litoral ubicado en el suroeste de 
Europa, entre España y Francia, en el límite de la península ibérica. Se constituye en Estado independiente, de 
derecho, democrático y social, cuya forma de gobierno es el coprincipado parlamentario. Su territorio está 
organizado en siete parroquias, con una población total de 79 877 habitantes a 28 de febrero de 2022. Su capital es
Andorra la Vieja. Con sus 468 km² de extensión territorial, Andorra es el micro-Estado más grande de Europa y está 
situado en los Pirineos, entre España y Francia; tiene una altitud media de 1996ms.n.m. Limita por el sur con 
España —con la provincia catalana de Lérida— y por el norte con Francia —con los departamentos de Ariège y Pirineos
Orientales (Occitania)—. Pertenece culturalmente a la Europa latina. Su sistema político es una democracia 
parlamentaria cuyos jefes de Estado son los copríncipes de Andorra: el obispo de Urgel y el presidente

In [ ]:
%%time

wiki_file_path = f"{TOKENIZERS_DATA_PATH}/wikipedia_es_plain.txt"
with open(wiki_file_path, "w", encoding="utf-8") as f:
    for article in dataset.take(1000):
        f.write(preprocess_text(article["text"]))
        f.write("\n")

In [1]:
!head -n 10 {TOKENIZERS_DATA_PATH}/wikipedia_es_plain.txt

head: cannot open '{TOKENIZERS_DATA_PATH}/wikipedia_es_plain.txt' for reading: No such file or directory


### Entrenando nuestro modelo con BPE

![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmedia1.tenor.com%2Fimages%2Fd565618bb1217a7c435579d9172270d0%2Ftenor.gif%3Fitemid%3D3379322&f=1&nofb=1&ipt=9719714edb643995ce9d978c8bab77f5310204960093070e37e183d5372096d9&ipo=images)

In [12]:
!pip install subword-nmt

In [ ]:
!ls {TOKENIZERS_DATA_PATH}

In [14]:
!subword-nmt --help

usage: subword-nmt [-h]
                   {learn-bpe,apply-bpe,get-vocab,learn-joint-bpe-and-vocab}
                   ...

subword-nmt: unsupervised word segmentation for neural machine translation and text generation 

positional arguments:
  {learn-bpe,apply-bpe,get-vocab,learn-joint-bpe-and-vocab}
                        command to run. Run one of the commands with '-h' for more info.
                        
                        learn-bpe: learn BPE merge operations on input text.
                        apply-bpe: apply given BPE operations to input text.
                        get-vocab: extract vocabulary and word frequencies from input text.
                        learn-joint-bpe-and-vocab: executes recommended workflow for joint BPE.

options:
  -h, --help            show this help message and exit


In [15]:
!subword-nmt learn-bpe --help

usage: subword-nmt learn-bpe [-h] [--input PATH] [--output PATH]
                             [--symbols SYMBOLS] [--min-frequency FREQ]
                             [--dict-input] [--total-symbols]
                             [--num-workers NUM_WORKERS] [--verbose]

learn BPE-based word segmentation

options:
  -h, --help            show this help message and exit
  --input PATH, -i PATH
                        Input text (default: standard input).
  --output PATH, -o PATH
                        Output file for BPE codes (default: standard output)
  --symbols SYMBOLS, -s SYMBOLS
                        Create this many new symbols (each representing a
                        character n-gram) (default: 10000)
  --min-frequency FREQ  Stop if no symbol pair has frequency >= FREQ (default:
                        2)
  --dict-input          If set, input file is interpreted as a dictionary
                        where each line contains a word-count pair
  --total-symbols, -t   subtrac

In [ ]:
%%time

!subword-nmt learn-bpe --num-workers -1 -s 300 < \
 {TOKENIZERS_DATA_PATH}/wikipedia_es_plain.txt > \
  {TOKENIZERS_MODEL_PATH}/wiki_es_300.model

100% 300/300 [00:12<00:00, 23.40it/s]
CPU times: user 95.3 ms, sys: 16.4 ms, total: 112 ms
Wall time: 17.5 s


In [ ]:
!echo "ando haciendo un análisis para claramente ver si puedes procesar esta oración mano" \
| subword-nmt apply-bpe -c {TOKENIZERS_MODEL_PATH}/wiki_es_300.model

ando h@@ aci@@ en@@ do un an@@ á@@ l@@ is@@ i@@ s para cl@@ ar@@ amente v@@ er s@@ i pu@@ ed@@ es pro@@ c@@ es@@ ar esta or@@ ación man@@ o


In [ ]:
%%time

!subword-nmt learn-bpe --num-workers -1 -s 10000 < \
{TOKENIZERS_DATA_PATH}/wikipedia_es_plain.txt > \
 {TOKENIZERS_MODEL_PATH}/wiki_es_10k.model

100% 10000/10000 [01:03<00:00, 157.69it/s]
CPU times: user 459 ms, sys: 70.4 ms, total: 529 ms
Wall time: 1min 10s


In [ ]:
!echo "ando haciendo un análisis para claramente ver si puedes procesar esta oración mano" \
| subword-nmt apply-bpe -c {TOKENIZERS_MODEL_PATH}/wiki_es_10k.model

ando haciendo un análisis para claramente ver si pued@@ es proces@@ ar esta or@@ ación mano


In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(
    " ".join(
        tokenizer.tokenize(
            "ando haciendo un análisis para claramente ver si puedes procesar esta oración mano"
        )
    ).replace("Ġ", "@@")
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ando @@h aci endo @@un @@an Ã¡ lis is @@para @@clar ament e @@ver @@si @@p ued es @@pro ces ar @@est a @@or aci Ã³n @@man o


### Aplicandolo a otros corpus: La biblia 📖🇻🇦

In [24]:
BIBLE_FILE_NAMES = {
    "spa": "spa-x-bible-reinavaleracontemporanea",
    "eng": "eng-x-bible-kingjames",
}

In [25]:
import requests


def get_bible_corpus(lang: str) -> str:
    """Download bible file corpus from GitHub repo"""
    file_name = BIBLE_FILE_NAMES[lang]
    r = requests.get(
        f"https://raw.githubusercontent.com/ximenina/theturningpoint/main/Detailed/corpora/corpusPBC/{file_name}.txt.clean.txt"
    )
    return r.text


def write_plain_text_corpus(raw_text: str, file_name: str) -> None:
    """Write file text on disk"""
    with open(f"{file_name}.txt", "w") as f:
        f.write(raw_text)

#### Tokenizando biblia en español

In [26]:
spa_bible_raw = get_bible_corpus("spa")
spa_bible_plain_text = preprocess_text(spa_bible_raw)

In [27]:
write_plain_text_corpus(spa_bible_plain_text, f"{CORPORA_PATH}/bible-spa")

In [ ]:
!subword-nmt apply-bpe -c {TOKENIZERS_MODEL_PATH}/wiki_es_10k.model < \
 {TOKENIZERS_DATA_PATH}/bible-spa.txt > \
 {TOKENIZERS_DATA_PATH}/bible-spa-tokenized.txt

#### Comparando resultados

In [31]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [32]:
spa_bible_words = word_tokenize(spa_bible_plain_text)

In [33]:
spa_bible_words[:10]

['principio',
 'del',
 'evangelio',
 'de',
 'jesucristo',
 ',',
 'el',
 'hijo',
 'de',
 'dios']

In [34]:
len(spa_bible_words)

30073

In [35]:
spa_bible_types = Counter(spa_bible_words)
len(spa_bible_types)

3317

In [36]:
spa_bible_types.most_common(30)

[(',', 1946),
 ('y', 1169),
 ('.', 1099),
 ('de', 1009),
 ('que', 927),
 ('a', 858),
 ('los', 645),
 ('la', 599),
 ('el', 572),
 (':', 539),
 ('se', 489),
 ('en', 461),
 ('«', 423),
 ('»', 423),
 ('jesús', 422),
 ('lo', 367),
 ('no', 312),
 ('le', 293),
 ('les', 267),
 ('dijo', 252),
 ('con', 220),
 ('pero', 217),
 ('al', 214),
 ('¿', 196),
 ('?', 195),
 ('por', 194),
 ('para', 172),
 ('su', 171),
 ('del', 165),
 ('un', 159)]

In [ ]:
with open(f"{TOKENIZERS_DATA_PATH}/bible-spa-tokenized.txt", "r") as f:
    tokenized_text = f.read()
spa_bible_tokenized = tokenized_text.split()

In [38]:
spa_bible_tokenized[:10]

['principio',
 'del',
 'evangelio',
 'de',
 'jesu@@',
 'cristo',
 ',',
 'el',
 'hijo',
 'de']

In [39]:
len(spa_bible_tokenized)

38497

In [40]:
spa_bible_tokenized_types = Counter(spa_bible_tokenized)
len(spa_bible_tokenized_types)

2297

In [41]:
spa_bible_tokenized_types.most_common(40)

[(',', 1946),
 ('y', 1209),
 ('.', 1099),
 ('de', 1038),
 ('a', 1000),
 ('que', 929),
 ('los', 736),
 ('la', 629),
 ('en', 586),
 ('el', 583),
 (':', 539),
 ('se', 526),
 ('«', 423),
 ('»', 423),
 ('jesús', 422),
 ('lo', 414),
 ('es', 386),
 ('no', 322),
 ('le', 320),
 ('les', 290),
 ('dijo', 258),
 ('ó', 221),
 ('con', 220),
 ('pero', 217),
 ('al', 216),
 ('¿', 196),
 ('?', 195),
 ('por', 194),
 ('o', 192),
 ('para', 175),
 ('su', 171),
 ('del', 165),
 ('un', 159),
 ('aron', 158),
 ('¡', 158),
 ('!', 158),
 ('an', 154),
 ('cuando', 148),
 ('ed@@', 147),
 ('las', 145)]

In [42]:
rprint("Biblia español")
rprint(f"Tipos ([bright_magenta]word-base[/]): {len(spa_bible_types)}")
rprint(f"Tipos ([bright_green]sub-word[/]): {len(spa_bible_tokenized_types)}")

Biblia español

Tipos (word-base): 3317

Tipos (sub-word): 2297

#### OOV: out of vocabulary

Palabras que se vieron en el entrenamiento pero no estan en el test

In [43]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    spa_bible_words, test_size=0.3, random_state=42
)
rprint(len(train_data), len(test_data))

21051 9022

In [44]:
s_1 = {"a", "b", "c", "d", "e"}
s_2 = {"a", "x", "y", "d"}
rprint(s_1 - s_2)
rprint(s_2 - s_1)

{'b', 'e', 'c'}

{'y', 'x'}

In [45]:
oov_test = set(test_data) - set(train_data)

In [46]:
for word in list(oov_test)[:3]:
    rprint(f"{word} in train: {word in set(train_data)}")

insistía in train: False

despedirme in train: False

venda in train: False

In [48]:
train_tokenized, test_tokenized = train_test_split(
    spa_bible_tokenized, test_size=0.3, random_state=42
)
rprint(len(train_tokenized), len(test_tokenized))

26947 11550

In [49]:
oov_tokenized_test = set(test_tokenized) - set(train_tokenized)

In [50]:
rprint("OOV ([bright_magenta]word-base[/]):", len(oov_test))
rprint("OOV ([bright_green]sub-word[/]):", len(oov_tokenized_test))

OOV (word-base): 533

OOV (sub-word): 216

### Type-token Ratio (TTR)

- Una forma de medir la variación del vocabulario en un corpus
- Este se calcula como $TTR = \frac{len(types)}{len(tokens)}$
- Puede ser útil para monitorear la variación lexica de un texto

In [51]:
stemmer = SnowballStemmer("spanish")
spa_bible_stemmed = [stemmer.stem(word) for word in spa_bible_words]
spa_bible_stemmed_types = set(spa_bible_stemmed)

In [52]:
rprint("Bible Spanish Information")
rprint("Tokens:", len(spa_bible_words))
rprint("Types ([bright_magenta]word-base[/]):", len(spa_bible_types))
rprint("Types ([bright_yellow]stemmed[/])", len(spa_bible_stemmed_types))
rprint("Types ([bright_green]BPE[/]):", len(spa_bible_tokenized_types))
rprint(
    "TTR ([bright_magenta]word-base[/]):", len(spa_bible_types) / len(spa_bible_words)
)
rprint(
    "TTR ([bright_yellow]stemmed[/]):",
    len(spa_bible_stemmed_types) / len(spa_bible_stemmed),
)
rprint(
    "TTR ([bright_green]BPE[/]):",
    len(spa_bible_tokenized_types) / len(spa_bible_tokenized),
)

Bible Spanish Information

Tokens: 30073

Types (word-base): 3317

Types (stemmed) 1897

Types (BPE): 2297

TTR (word-base): 0.11029827419944802

TTR (stemmed): 0.0630798390582915

TTR (BPE): 0.05966698703795101

## Word Embeddings (W2V)

Vamos a entrenar nuestras propias representaciones vectoriales utilizando la biblioteca [Gensim](https://radimrehurek.com/gensim/).

In [11]:
!pip install gensim

![we](https://miro.medium.com/v2/resize:fit:2000/1*SYiW1MUZul1NvL1kc1RxwQ.png)

### Datos: Noticias en Español

In [55]:
!pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [5]:
news_databuilder = load_dataset_builder("LeoCordoba/CC-NEWS-ES", "mx")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
rprint(news_databuilder.info)

DatasetInfo(
    description='',
    citation=' ',
    homepage='',
    license='',
    features={
        'country': Value(dtype='string', id=None),
        'text': Value(dtype='string', id=None),
        'id': Value(dtype='int32', id=None)
    },
    post_processed=None,
    supervised_keys=None,
    builder_name='parquet',
    dataset_name='cc-news-es',
    config_name='mx',
    version='0.0.0',
    splits={
        'train': SplitInfo(
            name='train',
            num_bytes=919053434,
            num_examples=652418,
            shard_lengths=[354000, 298418],
            dataset_name='cc-news-es'
        )
    },
    download_checksums={
        'https://huggingface.co/datasets/LeoCordoba/CC-NEWS-ES/resolve/main/mx.zip': {
            'num_bytes': 314518785,
            'checksum': None
        }
    },
    download_size=314518785,
    post_processing_size=None,
    dataset_size=919053434,
    size_in_bytes=1233572219
)

In [7]:
news_dataset = load_dataset(
    "LeoCordoba/CC-NEWS-ES", "mx", split="train", streaming=True
)

In [8]:
for post in news_dataset.take(1):
    rprint(post["text"])

La procuradora general de Justicia, Ernestina Godoy, informó que se logró imputar el delito de violación agravada a
cuatro policías preventivos que habrían abusado sexualmente de una menor de edad a bordo de una patrulla en calles 
de la demarcación Azcapotzalco; sin embargo, el secretario de Seguridad Ciudadana, Jesús Orta Martínez, dijo que la
dependencia a su cargo aún no ha sido notificada.    Por la tarde, el jefe de la policía capitalina, luego de un 
encuentro con el presidente de la Cámara Nacional de Comercio, Servicios y Turismo de la Ciudad de México, Nathan 
Poplawsky, comentó que los uniformados siguen en resguardo por la Dirección General de Asuntos Internos y no han 
sido suspendidos.

In [13]:
from gensim.utils import simple_preprocess

print(simple_preprocess(post["text"], deacc=True)[:10])

['la', 'procuradora', 'general', 'de', 'justicia', 'ernestina', 'godoy', 'informo', 'que', 'se']


In [15]:
from tqdm.notebook import tqdm
from datasets import load_dataset
from gensim.utils import simple_preprocess


class CCNewsExtractor:
    """
    Iterador optimizado para CC-NEWS-ES + Word2Vec.
    Diseñado para alta velocidad y compatibilidad con los epochs de Gensim.
    """

    def __init__(self, lang: str = "mx", max_posts: int = -1):
        self.dataset = load_dataset(
            "LeoCordoba/CC-NEWS-ES", name=lang, split="train", streaming=True
        )
        self.max_posts = max_posts

    def __iter__(self):
        for item in tqdm(self.dataset.take(self.max_posts)):
            text = item.get("text", "")
            if not text:
                continue

            words = simple_preprocess(text, deacc=False, min_len=6)

            if not words:
                continue
            yield words

In [16]:
# Uso con tu función train_model
iterator = CCNewsExtractor(lang="mx", max_posts=3)

In [17]:
for i in iterator:
    rprint(i[:10])

0it [00:00, ?it/s]

[
    'procuradora',
    'general',
    'justicia',
    'ernestina',
    'informó',
    'imputar',
    'delito',
    'violación',
    'agravada',
    'cuatro'
]

['réplica', 'gobernador', 'licencia', 'espetó', 'quiero', 'precisarle', 'compañera', 'ortega', 'miente', 'mientes']

[
    'estadunidense',
    'claire',
    'rasmus',
    'primero',
    'títulos',
    'disputaron',
    'natación',
    'conquistar',
    'metros',
    'femenil'
]

In [18]:
%%time
sentences = CCNewsExtractor(lang="mx", max_posts=10)

CPU times: user 31 ms, sys: 4.57 ms, total: 35.6 ms
Wall time: 732 ms


In [19]:
for sentence in sentences:
    print(sentence)

0it [00:00, ?it/s]

['procuradora', 'general', 'justicia', 'ernestina', 'informó', 'imputar', 'delito', 'violación', 'agravada', 'cuatro', 'policías', 'preventivos', 'habrían', 'abusado', 'sexualmente', 'patrulla', 'calles', 'demarcación', 'azcapotzalco', 'embargo', 'secretario', 'seguridad', 'ciudadana', 'martínez', 'dependencia', 'notificada', 'policía', 'capitalina', 'encuentro', 'presidente', 'cámara', 'nacional', 'comercio', 'servicios', 'turismo', 'ciudad', 'méxico', 'nathan', 'poplawsky', 'comentó', 'uniformados', 'siguen', 'resguardo', 'dirección', 'general', 'asuntos', 'internos', 'suspendidos']
['réplica', 'gobernador', 'licencia', 'espetó', 'quiero', 'precisarle', 'compañera', 'ortega', 'miente', 'mientes', 'mientes', 'ivonne', 'conocemos', 'yucatán', 'estado', 'seguro', 'revisemos', 'cifras', 'sistema', 'nacional', 'seguridad', 'pública', 'campeche', 'entidad', 'índice', 'delictivo', 'lorena', 'candidata', 'enarbola', 'bandera', 'derecho', 'juventud', 'llegar', 'presidencia', 'trataba', 'meter

In [20]:
from gensim.models import word2vec

In [24]:
EMB_MODELS_DIR = os.path.join(MODELS_PATH, "embeddings")

os.makedirs(EMB_MODELS_DIR, exist_ok=True)

In [25]:
from enum import Enum


class Algorithms(Enum):
    CBOW = 0
    SKIP_GRAM = 1

In [26]:
def load_model(model_path: str):
    """Load a word2vec model from a given path.
    """
    try:
        print(model_path)
        return word2vec.Word2Vec.load(model_path)
    except FileNotFoundError:
        print(f"[WARN] Model not found in path {model_path}")
        return None

In [28]:
def train_model(
    sentences: list,
    model_name: str,
    vector_size: int,
    window=5,
    workers=2,
    algorithm=Algorithms.CBOW,
):
    model_name_params = f"{model_name}-vs{vector_size}-w{window}-{algorithm.name}.model"
    model_path = os.path.join(EMB_MODELS_DIR, model_name_params)
    if load_model(model_path) is not None:
        print(f"Already exists the model {model_path}")
        return load_model(model_path)
    print(f"TRAINING: {model_path}")
    if algorithm in [Algorithms.CBOW, Algorithms.SKIP_GRAM]:
        model = word2vec.Word2Vec(
            sentences,
            vector_size=vector_size,
            window=window,
            workers=workers,
            sg=algorithm.value,
            seed=42,
        )
    else:
        print("[ERROR] algorithm not implemented yet :p")
        return
    try:
        model.save(model_path)
    except:
        print(f"[ERROR] Saving model at {model_path}")
    return model

### CBOW

In [29]:
skipm_gram_model = load_model(
    os.path.join(EMB_MODELS_DIR, "eswiki-xl-300-SKIP_GRAM.model")
)

./models/embeddings/eswiki-xl-300-SKIP_GRAM.model
[WARN] Model not found in path ./models/embeddings/eswiki-xl-300-SKIP_GRAM.model


In [30]:
%%time
cbow_model = train_model(
    CCNewsExtractor(lang="mx", max_posts=100_000),
    "es_news_hf",
    vector_size=100,
    window=3,
    workers=2,
    algorithm=Algorithms.CBOW,
)

./models/embeddings/es_news_hf-vs100-w3-CBOW.model
[WARN] Model not found in path ./models/embeddings/es_news_hf-vs100-w3-CBOW.model
TRAINING: ./models/embeddings/es_news_hf-vs100-w3-CBOW.model


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

CPU times: user 14min 34s, sys: 11.8 s, total: 14min 46s
Wall time: 13min 16s


### Skip gram

In [ ]:
%%time
skip_gram_model = train_model(
    CCNewsExtractor(lang="mx", max_posts=100_000),
    "es_news_hf",
    300,
    5,
    workers=12,
    algorithm=Algorithms.SKIP_GRAM,
)

In [31]:
def report_stats(model) -> None:
    """Print report of a model"""
    print(
        "Number of words in the corpus used for training the model: ",
        model.corpus_count,
    )
    print("Number of words in the model: ", len(model.wv.index_to_key))
    print("Time [s], required for training the model: ", model.total_train_time)
    print("Count of trainings performed to generate this model: ", model.train_count)
    print("Length of the word2vec vectors: ", model.vector_size)
    print("Applied context length for generating the model: ", model.window)

In [32]:
report_stats(cbow_model)

Number of words in the corpus used for training the model:  100000
Number of words in the model:  61046
Time [s], required for training the model:  647.5827804889977
Count of trainings performed to generate this model:  1
Length of the word2vec vectors:  100
Applied context length for generating the model:  3


In [ ]:
report_stats(skip_gram_model)

### Operaciones con los vectores entrenados

Veremos operaciones comunes sobre vectores. Estos resultados dependeran del modelo que hayamos cargado en memoria

In [33]:
models = {
    Algorithms.CBOW: cbow_model,
    #Algorithms.SKIP_GRAM: skip_gram_model,
}

In [35]:
model = models[Algorithms.CBOW]

In [36]:
for index, word in enumerate(model.wv.index_to_key):
    if index == 100:
        break
    print(f"word #{index}/{len(model.wv.index_to_key)} is {word}")

word #0/61046 is méxico
word #1/61046 is personas
word #2/61046 is también
word #3/61046 is gobierno
word #4/61046 is estado
word #5/61046 is nacional
word #6/61046 is cuando
word #7/61046 is presidente
word #8/61046 is porque
word #9/61046 is millones
word #10/61046 is manera
word #11/61046 is contra
word #12/61046 is ciudad
word #13/61046 is información
word #14/61046 is durante
word #15/61046 is través
word #16/61046 is editora
word #17/61046 is derechos
word #18/61046 is además
word #19/61046 is estados
word #20/61046 is seguridad
word #21/61046 is cualquier
word #22/61046 is acuerdo
word #23/61046 is después
word #24/61046 is general
word #25/61046 is pasado
word #26/61046 is fueron
word #27/61046 is ciento
word #28/61046 is federal
word #29/61046 is unidos
word #30/61046 is trabajo
word #31/61046 is tiempo
word #32/61046 is presente
word #33/61046 is usuarios
word #34/61046 is mientras
word #35/61046 is política
word #36/61046 is momento
word #37/61046 is quienes
word #38/61046 i

In [39]:
gato_vec = model.wv["méxico"]
print(gato_vec[:10])
print(len(gato_vec))

[ 0.3655688   0.2082324   0.7913874  -1.2864966   0.58035654 -0.02981467
 -1.3454736   0.69980377 -0.26873645 -0.09159524]
100


In [40]:
try:
    agustisidad_vec = model.wv["agusticidad"]
except KeyError:
    print("OOV founded!")

OOV founded!


In [41]:
agustisidad_vec[:10]
len(agustisidad_vec)

NameError: name 'agustisidad_vec' is not defined

In [44]:
model.wv.most_similar("méxico", topn=5)

[('juárez', 0.5535541772842407),
 ('amurallada', 0.5224052667617798),
 ('mexicano', 0.502213716506958),
 ('querétaro', 0.4836301803588867),
 ('aeroportuario', 0.46319350600242615)]

Podemos ver como la similitud entre palabras decrece

In [48]:
word_pairs = [
    ("automóvil", "camión"),
    ("automóvil", "bicicleta"),
    ("automóvil", "cereal"),
    ("automóvil", "distrito"),
]

for w1, w2 in word_pairs:
    print(f"{w1} - {w2} {model.wv.similarity(w1, w2)}")

automóvil - camión 0.8299127221107483
automóvil - bicicleta 0.6629902124404907
automóvil - cereal 0.3022066354751587
automóvil - distrito -0.057788509875535965


In [49]:
# rey es a hombre como ___ a mujer
# londres es a inglaterra como ____ a vino
model.wv.most_similar(positive=["saltillo", "morelos"], negative=["cuernavaca"])

[('coahuila', 0.7187989950180054),
 ('torreón', 0.6777724623680115),
 ('aguascalientes', 0.631315290927887),
 ('durango', 0.6264187097549438),
 ('querétaro', 0.6211353540420532),
 ('laguna', 0.5898624658584595),
 ('arizpe', 0.5831578969955444),
 ('monclova', 0.5664711594581604),
 ('potosí', 0.5655553936958313),
 ('campeche', 0.5538091063499451)]

In [50]:
model.wv.doesnt_match(["disco", "música", "mantequilla", "cantante"])

'mantequilla'

In [51]:
model.wv.similarity("noche", "noches")

KeyError: "Key 'noche' not present"

#### Evaluación

`Word2Vec` es una tarea de entrenamiento semi-supervisada, por lo tanto, es difícil evaluar el desempeño de un modelo. La evaluación dependerá de la tarea.

Sin embargo, Google liberó un conjunto de evaluación con ejemplos semánticos/sintácticos. Se sigue la forma "A es a B como C es a D". Por ejemplo, "tokio es a japon como berlin es a alemania".

Se tienen varias categorias como comparaciones sintácticas, capitales, miembros de una familia, etc.

In [ ]:
from gensim.test.utils import datapath

model.wv.evaluate_word_analogies(datapath("questions-words.txt"))

## Modelos del Lenguaje Neuronales (Bengio)

- [(Bengio et al 2003)](https://dl.acm.org/doi/10.5555/944919.944966) proponen una arquitecura neuronal como alternativa a los modelos del lenguaje estadísticos
- Esta arquitectura lidia mejor con los casos donde las probabilidades se hacen cero, sin necesidad de aplicar una técnica de smoothing.

<p float="left">
  <img src="https://toppng.com/public/uploads/preview/at-the-movies-will-smith-meme-tada-11562851401lnexjqtwf9.png" width="100" />
  <img src="https://abhinavcreed13.github.io/assets/images/bengio-model.png" width="600"/>
</p>

In [1]:
def lm_preprocess_corpus(corpus: list[str]) -> list[str]:
    """Función de preprocesamiento para LM

    Esta función está diseñada para preprocesar
    corpus para modelos del lenguaje neuronales.
    Agrega tokens de inicio y fin, normaliza
    palabras a minusculas
    """
    preprocessed_corpus = []
    for sent in corpus:
        result = [word.lower() for word in sent]
        # Al final de la oración
        result.append("<EOS>")
        result.insert(0, "<BOS>")
        preprocessed_corpus.append(result)
    return preprocessed_corpus

In [2]:
def get_words_freqs(corpus: list[list[str]]):
    """Calcula la frecuencia de las palabras en un corpus"""
    words_freqs = {}
    for sentence in corpus:
        for word in sentence:
            words_freqs[word] = words_freqs.get(word, 0) + 1
    return words_freqs

In [3]:
UNK_LABEL = "<UNK>"


def get_words_indexes(words_freqs: dict) -> dict:
    """Calcula los indices de las palabras dadas sus frecuencias"""
    result = {}
    for idx, word in enumerate(words_freqs.keys()):
        # Happax legomena happends
        if words_freqs[word] == 1:
            # Temp index for unknowns
            result[UNK_LABEL] = len(words_freqs)
        else:
            result[word] = idx

    return {word: idx for idx, word in enumerate(result.keys())}, {
        idx: word for idx, word in enumerate(result.keys())
    }

In [4]:
import nltk

nltk.download("gutenberg")
nltk.download("abc")
nltk.download("genesis")
nltk.download("inaugural")
nltk.download("state_union")
nltk.download("webtext")
nltk.download("punkt_tab")

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package abc to /root/nltk_data...
[nltk_data]   Unzipping corpora/abc.zip.
[nltk_data] Downloading package genesis to /root/nltk_data...
[nltk_data]   Unzipping corpora/genesis.zip.
[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Unzipping corpora/inaugural.zip.
[nltk_data] Downloading package state_union to /root/nltk_data...
[nltk_data]   Unzipping corpora/state_union.zip.
[nltk_data] Downloading package webtext to /root/nltk_data...
[nltk_data]   Unzipping corpora/webtext.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
from nltk.corpus import abc, genesis, gutenberg, inaugural, state_union, webtext

# Exploración del corpus
total_sents = 0
corpora = []

plaintext_corpora = {
    "abc": abc,
    "Gutenberg": gutenberg,
    #"Genesis": genesis, Este no lo usamos por una buena razón
    "Inaugural": inaugural,
    "State Union": state_union,
    "Web": webtext,
}

for title, corpus in plaintext_corpora.items():
    corpus_sents = lm_preprocess_corpus(corpus.sents())
    corpus_len = len(corpus_sents)
    total_sents += corpus_len
    print(f"{title} sents={corpus_len}")
    corpora.extend(corpus_sents)
print(f"Total={total_sents}")

abc sents=29059
Gutenberg sents=98552
Inaugural sents=5395
State Union sents=17930
Web sents=25733
Total=176669


In [6]:
len(corpora)

176669

In [7]:
corpora[42]

['<BOS>',
 'awi',
 'is',
 'showing',
 'wool',
 'blend',
 't',
 '-',
 'shirts',
 'and',
 'casual',
 'wear',
 'to',
 'manufacturers',
 'this',
 'week',
 'at',
 'the',
 'largest',
 'trade',
 'show',
 'for',
 'the',
 'industry',
 'being',
 'held',
 'in',
 'germany',
 '.',
 '<EOS>']

In [8]:
words_freqs = get_words_freqs(corpora)

In [9]:
words_freqs["the"]

214361

In [10]:
len(words_freqs)

65763

In [11]:
count = 0
for word, freq in words_freqs.items():
    if freq == 1 and count <= 10:
        print(word, freq)
        count += 1

totalled 1
skimmed 1
ploy 1
multinationals 1
truckloads 1
conner 1
9m 1
elevators 1
devonport 1
macleod 1
stopwork 1


In [12]:
words_indexes, index_to_word = get_words_indexes(words_freqs)

In [13]:
words_indexes["god"]

9562

In [15]:
index_to_word[9562]

'god'

In [16]:
len(words_indexes)

41392

In [17]:
len(index_to_word)

41392

In [18]:
def get_word_id(words_indexes: dict, word: str) -> int:
    """Obtiene el id de una palabra dada

    Si no se encuentra la palabra se regresa el id
    del token UNK
    """
    unk_word_id = words_indexes[UNK_LABEL]
    return words_indexes.get(word, unk_word_id)

### Obtenemos trigramas

Convertiremos los trigramas obtenidos a secuencias de idx, y preparamos el conjunto de entrenamiento $x$ y $y$

- x: Contexto
- y: Predicción de la siguiente palabra

In [19]:
from nltk import ngrams


def get_train_test_data(
    corpus: list[list[str]], words_indexes: dict, n: int
) -> tuple[list, list]:
    """Obtiene el conjunto de train y test

    Requerido en el step de entrenamiento del modelo neuronal
    """
    x_train = []
    y_train = []
    for sent in corpus:
        n_grams = ngrams(sent, n)
        for w1, w2, w3 in n_grams:
            x_train.append(
                [get_word_id(words_indexes, w1), get_word_id(words_indexes, w2)]
            )
            y_train.append([get_word_id(words_indexes, w3)])
    return x_train, y_train

### Preparando Pytorch

$x' = e(x_1) \oplus e(x_2)$

$h = \tanh(W_1 x' + b)$

$y = softmax(W_2 h)$

In [20]:
# cargamos bibliotecas
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import time

In [21]:
# Setup de parametros
EMBEDDING_DIM = 200
CONTEXT_SIZE = 2
BATCH_SIZE = 256

H = 100
torch.manual_seed(42)
# Tamaño del Vocabulario
V = len(words_indexes)

In [22]:
x_train, y_train = get_train_test_data(corpora, words_indexes, n=3)

In [23]:
import numpy as np

train_set = np.concatenate((x_train, y_train), axis=1)
# partimos los datos de entrada en batches
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE)

### Creamos la arquitectura del modelo

In [24]:
# Trigram Neural Network Model
class TrigramModel(nn.Module):
    """Clase padre: https://pytorch.org/docs/stable/generated/torch.nn.Module.html"""

    def __init__(self, vocab_size, embedding_dim, context_size, h):
        super(TrigramModel, self).__init__()
        self.context_size = context_size
        self.embedding_dim = embedding_dim
        # TODO: Se aprenden los embeddings de aca?
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, h)
        self.linear2 = nn.Linear(h, vocab_size)

    def forward(self, inputs):
        # x': concatenation of x1 and x2 embeddings   -->
        # self.embeddings regresa un vector por cada uno de los índices que se les pase como entrada.
        # view() les cambia el tamaño para concatenarlos
        embeds = self.embeddings(inputs).view(
            (-1, self.context_size * self.embedding_dim)
        )
        # h: tanh(W_1.x' + b)  -->
        out = torch.tanh(self.linear1(embeds))
        # W_2.h                 -->
        out = self.linear2(out)
        # log_softmax(W_2.h)      -->
        # dim=1 para que opere sobre renglones, pues al usar batchs tenemos varios vectores de salida
        log_probs = F.log_softmax(out, dim=1)

        return log_probs

In [25]:
# Seleccionar la GPU si está disponible
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)

In [26]:
device

'cuda'

In [29]:
import os
NN_MODELS_PATH = os.path.join("models", "nn")

os.makedirs(NN_MODELS_PATH, exist_ok=True)

LM_PATH = os.path.join(NN_MODELS_PATH, "trigrams_nlm_cpu_epoch3.pt")

In [34]:
print(f"Training on device {device}")

# 1. Pérdida. Negative log-likelihood loss
loss_function = nn.NLLLoss()

# 2. Instanciar el modelo y enviarlo a device
model = TrigramModel(V, EMBEDDING_DIM, CONTEXT_SIZE, H).to(device)

# 3. Optimización. ADAM optimizer
optimizer = optim.Adam(model.parameters(), lr=2e-3)

# ------------------------- TRAIN & SAVE MODEL ------------------------
EPOCHS = 5
for epoch in range(EPOCHS):
    st = time.time()
    print("\n--- Training model Epoch: {} ---".format(epoch))
    for it, data_tensor in enumerate(train_loader):
        # Mover los datos al dispositivo
        context_tensor = data_tensor[:, 0:2].to(device)
        target_tensor = data_tensor[:, 2].to(device)

        # Resetamos los gradientes de la iteración anterior
        model.zero_grad()

        # FORWARD:
        log_probs = model(context_tensor)

        # compute loss function
        loss = loss_function(log_probs, target_tensor)

        # BACKWARD:
        loss.backward()
        optimizer.step()

        if it % 500 == 0:
            print(
                "Training Iteration {} of epoch {} complete. Loss: {}; Time taken (s): {}".format(
                    it, epoch, loss.item(), (time.time() - st)
                )
            )
            st = time.time()

    # saving model
    model_path = os.path.join(
        NN_MODELS_PATH, f"lm_large_{device}_context_{CONTEXT_SIZE}_epoch_{epoch}.dat"
    )
    torch.save(model.state_dict(), model_path)
    print(f"Model saved for epoch={epoch} at {model_path}")

Training on device cuda

--- Training model Epoch: 0 ---
Training Iteration 0 of epoch 0 complete. Loss: 10.685422897338867; Time taken (s): 0.011324882507324219
Training Iteration 500 of epoch 0 complete. Loss: 6.634129524230957; Time taken (s): 3.860532522201538
Training Iteration 1000 of epoch 0 complete. Loss: 5.202940940856934; Time taken (s): 3.8278019428253174
Training Iteration 1500 of epoch 0 complete. Loss: 6.4298529624938965; Time taken (s): 3.8783037662506104
Training Iteration 2000 of epoch 0 complete. Loss: 6.199047088623047; Time taken (s): 3.8573617935180664
Training Iteration 2500 of epoch 0 complete. Loss: 5.583111763000488; Time taken (s): 3.85878324508667
Training Iteration 3000 of epoch 0 complete. Loss: 6.950370788574219; Time taken (s): 3.8885746002197266
Training Iteration 3500 of epoch 0 complete. Loss: 5.005373954772949; Time taken (s): 3.9056472778320312
Training Iteration 4000 of epoch 0 complete. Loss: 4.98941707611084; Time taken (s): 3.890758991241455


KeyboardInterrupt: 

In [32]:
model

TrigramModel(
  (embeddings): Embedding(41392, 200)
  (linear1): Linear(in_features=400, out_features=100, bias=True)
  (linear2): Linear(in_features=100, out_features=41392, bias=True)
)

In [35]:
def get_torch_model(path: str) -> TrigramModel:
    """Obtiene modelo de pytorch desde disco"""
    model_loaded = TrigramModel(V, EMBEDDING_DIM, CONTEXT_SIZE, H)
    model_loaded.load_state_dict(torch.load(path))
    model_loaded.eval()
    return model_loaded

In [41]:
model = get_torch_model(os.path.join("models/nn/", "lm_large_cuda_context_2_epoch_0.dat"))

In [43]:
device

'cuda'

In [46]:
W1 = "<BOS>"
W2 = "my"

IDX1 = get_word_id(words_indexes, W1)
IDX2 = get_word_id(words_indexes, W2)

# Obtenemos Log probabidades p(W3|W2,W1)
probs = model(torch.tensor([[IDX1, IDX2]]).to("cpu")).detach().tolist()

In [47]:
len(probs[0])

41392

In [48]:
# Creamos diccionario con {idx: logprob}
model_probs = {}
for idx, p in enumerate(probs[0]):
    model_probs[idx] = p

# Sort:
model_probs_sorted = sorted(
    ((prob, idx) for idx, prob in model_probs.items()), reverse=True
)

# Printing word  and prob (retrieving the idx):
topcandidates = 0
for prob, idx in model_probs_sorted:
    # Retrieve the word associated with that idx
    word = index_to_word[idx]
    print(idx, word, prob, np.exp(prob))

    topcandidates += 1

    if topcandidates > 10:
        break

17630 palate -3.079512119293213 0.04598168475433661
377 first -3.583524703979492 0.02777761762481218
89 <UNK> -3.6454319953918457 0.02611012796227088
7207 impression -3.768012523651123 0.02309792427652138
3323 nose -3.8374762535095215 0.021547914253230074
784 family -4.006741046905518 0.018192587521372018
8096 father -4.117477893829346 0.016285536541363034
2711 finish -4.1512837409973145 0.015744192041365245
2935 own -4.221234321594238 0.014680512843036278
1232 head -4.257209777832031 0.014161761796037765
1218 good -4.323977470397949 0.013247088713893745


In [49]:
print(index_to_word.get(model_probs_sorted[0][1]))

palate


### Generacion de lenguaje

In [52]:
def get_likely_words(
    model: TrigramModel,
    context: str,
    words_indexes: dict,
    index_to_word: dict,
) -> list[tuple]:
    """Dado un contexto obtiene las palabras más probables
    """
    model_probs = {}
    words = context.split()
    idx_word_1 = get_word_id(words_indexes, words[0])
    idx_word_2 = get_word_id(words_indexes, words[1])
    probs = model(torch.tensor([[idx_word_1, idx_word_2]]).to("cpu")).detach().tolist()

    for idx, p in enumerate(probs[0]):
        model_probs[idx] = p

    # Strategy: Sort and get top-K words to generate text
    return sorted(
        ((prob, index_to_word[idx]) for idx, prob in model_probs.items()), reverse=True
    )

In [53]:
sentence = "this is"
get_likely_words(model, sentence, words_indexes, index_to_word)[:3]

[(-2.2015092372894287, 'a'),
 (-2.5466535091400146, 'very'),
 (-2.741962194442749, 'not')]

In [56]:
import random
from random import randint

def get_next_top_p_word(words: list[tuple[float, str]], p: float = 0.8) -> str:
    """
    Selecciona la siguiente palabra utilizando Nucleus Sampling (Top-p).

    Params:
    - words: Lista de tuplas (palabra, probabilidad).
    - p: Umbral de masa de probabilidad acumulada (típicamente entre 0.8 y 0.95).
    """
    if not words:
        return "<EOS>"

    # Aseguramos que la lista esté ordenada de mayor a menor probabilidad
    # sorted_words = sorted(words, key=lambda x: x[1], reverse=True)

    valid_words = []
    valid_probs = []
    cumulative_prob = 0.0

    # Recolectamos palabras hasta que la suma de probabilidades alcance el umbral 'p'
    for log_prob, word in words:
        # Convertimos log_prob a probabilidad normal
        prob = np.exp(log_prob)
        valid_words.append(word)
        valid_probs.append(prob)
        cumulative_prob += prob

        if cumulative_prob >= p:
            break

    # Muestreamos una palabra del subconjunto válido (núcleo) usando sus probabilidades como pesos.
    # random.choices devuelve una lista, por lo que extraemos el elemento [0]
    return random.choices(valid_words, weights=valid_probs, k=1)[0]


def get_next_word(words: list[tuple[float, str]]) -> str:
    # From a top-K list of words get a random word
    return words[randint(0, len(words) - 1)][1]

In [57]:
get_next_top_p_word(get_likely_words(model, sentence, words_indexes, index_to_word))

'better'

In [58]:
MAX_TOKENS = 50
TOP_P = 0.7


def generate_text(
    model: TrigramModel,
    history: str,
    words_indexes: dict,
    index_to_word: dict,
    tokens_count: int = 0,
) -> None:
    next_word = get_next_top_p_word(
        get_likely_words(
            model, history, words_indexes, index_to_word
        ), p=TOP_P
    )
    print(next_word, end=" ")
    tokens_count += 1
    if tokens_count == MAX_TOKENS or next_word == "<EOS>":
        return
    generate_text(
        model,
        history.split()[1] + " " + next_word,
        words_indexes,
        index_to_word,
        tokens_count,
    )

In [59]:
sentence = "god tells"
print(sentence, end=" ")
generate_text(model, sentence, words_indexes, index_to_word)

god tells the grape - not very good but the other wines that ? <EOS> 

# Práctica 4: Evaluación de modelos del lenguaje neuronales

**Fecha: 5 de Mayo 2026 11:59pm**

### Formáto de entrega
- Crear una carpeta con el nombre de su equipo dentro de `practicas/`
- Incluir los archivos requeridos (notebook, script Python, README)
- Ejemplo de estructura:

```
practicas/
├── krustaceo/
│   └── P4
│       ├── mi_practica4.ipynb
│       ├── mi_practica4.py
│       └── README.md  # <-- Incluir los nombres de los integrantes
```

#### Investigación

La calidad de un modelo del lenguaje puede ser evaluado por medio de su perplejidad (perplexity)

- Investigar como calcular la perplejidad de un modelo del lenguaje y como evaluarlo con esa medida
    - Incluir en el `README.md` de su entrega una síntesis de esta investigación. Sean breves
        - Explicación clara de qué es la **perplejidad** (perplexity) y cómo se calcula
        - Fórmula matemática con explicación de cada componente
        - Relación entre perplejidad y calidad del modelo
        - Ventajas y limitaciones de esta métrica
- Evalua el modelo entrenado en clase con los corpus de `nltk`
    - Descarga el modelo acá

#### Creación de modelos del lenguaje

- Entrena un nuevo modelo del lenguaje neuronal con los corpus de `nltk` aplicando previamente sub-word tokenization a los corpus
    - Puedes utilizar un modelo de tokenizacion pre-entrenado o entrenar uno desde cero
    - Utiliza el corpus `genesis` de `nltk` como test de evaluación.
- Evalua tu modelo calculando su perplejidad.


#### Análisis comparativo

- Realizar un análisis comparativo entre ambos modelos.

| Métrica               | Modelo Base | Modelo Subword |
|-----------------------|-------------|----------------|
| Perplejidad (genesis) |             |                |
| Tamaño vocabulario    |             |                |
| OOV Rate              |             |                |

- Incluir en el `README.md`:
    - Discusión sobre qué modelo tuvo mejor desempeño y por qué
    - Ventajas y desventajas de cada enfoque
    - Recomendaciones para mejorar ambos modelos


**NOTA:** Sube tu modelo a alguna plataforma de almacenamiento (Google Drive, Nextcloud, Hugging Face, etc), proporciona el link de descarga y el código para cargar el modelo en memoria. **No subas tu modelo al repositorio de GitHub**.

## EXTRA

- Diseña una estrategia de generación de usando el modelo del lenguaje entrenado con sub-word tokenization
- Se deben generar secuencias de palabras (no subwords)
- Muestra tres ejemplos de generación